In [0]:
UC_CATALOG = "preethiworkspace_7405608993331015"
UC_SCHEMA  = "default"
VOLUME_NAME = "bank_fraud"

BASE = f"/Volumes/preethiworkspace_7405608993331015/default/bank_fraud"

paths = {
  "landing":      f"{BASE}/landing/transactions",
  "schemaLoc":    f"{BASE}/schema/transactions",
  "ckpt_bronze":  f"{BASE}/checkpoints/bronze",
  "ckpt_silver":  f"{BASE}/checkpoints/silver",
  "ckpt_gold":    f"{BASE}/checkpoints/gold",
  "reference":    f"{BASE}/reference"
}

tables = {
  "bronze": f"{UC_CATALOG}.{UC_SCHEMA}.bronze_transactions",
  "silver": f"{UC_CATALOG}.{UC_SCHEMA}.silver_transactions",
  "gold":   f"{UC_CATALOG}.{UC_SCHEMA}.gold_alerts",
  "stats":  f"{UC_CATALOG}.{UC_SCHEMA}.dim_category_stats"
}

# quick sanity check
display(dbutils.fs.ls(BASE))
paths, tables

In [0]:
import pyspark.sql.functions as F

silver_stream = spark.readStream.table(tables["silver"])
stats_static  = spark.table(tables["stats"])

gold_df = (
  silver_stream.join(stats_static, on="category", how="left")
    .withColumn("z_amt",
        F.when(F.col("std_amt_cat").isNull() | (F.col("std_amt_cat") == 0), F.lit(0.0))
         .otherwise((F.col("amt") - F.col("avg_amt_cat")) / F.col("std_amt_cat"))
    )
    .withColumn("odd_hour_flag", F.when((F.col("hour") < 6) | (F.col("hour") > 22), 1).otherwise(0))
    .withColumn("high_amt_flag", F.when(F.col("z_amt") > 3, 1).otherwise(0))
    .withColumn("risk_score",
        (F.when(F.col("z_amt") < -5, -5).when(F.col("z_amt") > 10, 10).otherwise(F.col("z_amt")) * 0.6) +
        (F.col("odd_hour_flag") * 1.0) +
        (F.col("high_amt_flag") * 2.0)
    )
    .withColumn("anomaly_flag", F.when(F.col("risk_score") >= 3.0, 1).otherwise(0))
    .withColumn("alert_type",
        F.when(F.col("risk_score") >= 6, F.lit("HIGH"))
         .when(F.col("risk_score") >= 3, F.lit("MEDIUM"))
         .otherwise(F.lit("LOW"))
    )
    .withColumn("_gold_write_ts", F.current_timestamp())
)

(
  gold_df.writeStream
    .format("delta")
    .option("checkpointLocation", paths["ckpt_gold"])
    .outputMode("append")
    .trigger(availableNow=True)
    .table(tables["gold"])
)

In [0]:
%sql
select count(*) from default.gold_alerts